In [1]:
import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve
from sklearn.preprocessing import StandardScaler
 
 
FEATURE_DIR  = r"E:/Python/MSc_Project_Upgrade/datasets/bdt"
BASE_DIR     = r"E:/Python/MSc_Project_Upgrade/results_analysis/bdt"
COMPARE_DIR  = os.path.join(BASE_DIR, "comparison")
os.makedirs(COMPARE_DIR, exist_ok=True)

In [2]:
TEST_ENERGIES   = [100, 125, 150, 200, 250, 300]
WORKING_POINTS  = [0.30, 0.50, 0.70, 0.90]
FEATURE_NAMES   = [
    "pt", "mass", "ncharged", "nneutrals",
    "ehad", "chf", "nef",
    "tau1", "tau21", "tau32",
]
 
MODEL_CONFIGS = {
    "rf_125": {
        "type": "rf", "train_energy": 125,
        "results_dir": os.path.join(BASE_DIR, "RF_model1_125GeV"),
        "model_file":  "rf_model1_125GeV.pkl",
        "scaler_file": "rf_model1_scaler.npz",
        "label": "Random Forest", "color": "royalblue",   "marker": "o", "ls": "-",
    },
    "xgb_125": {
        "type": "xgb", "train_energy": 125,
        "results_dir": os.path.join(BASE_DIR, "XGB_model1_125GeV"),
        "model_file":  "xgb_model1_125GeV.json",
        "scaler_file": "xgb_model1_scaler.npz",
        "label": "XGBoost",       "color": "darkorange",  "marker": "s", "ls": "--",
    },
    "lgbm_125": {
        "type": "lgbm", "train_energy": 125,
        "results_dir": os.path.join(BASE_DIR, "LGBM_model1_125GeV"),
        "model_file":  "lgbm_model1_125GeV.txt",
        "scaler_file": "lgbm_model1_scaler.npz",
        "label": "LightGBM",      "color": "forestgreen", "marker": "^", "ls": "-.",
    },
    "rf_250": {
        "type": "rf", "train_energy": 250,
        "results_dir": os.path.join(BASE_DIR, "RF_model2_250GeV"),
        "model_file":  "rf_model2_250GeV.pkl",
        "scaler_file": "rf_model2_scaler.npz",
        "label": "Random Forest", "color": "royalblue",   "marker": "o", "ls": "-",
    },
    "xgb_250": {
        "type": "xgb", "train_energy": 250,
        "results_dir": os.path.join(BASE_DIR, "XGB_model2_250GeV"),
        "model_file":  "xgb_model2_250GeV.json",
        "scaler_file": "xgb_model2_scaler.npz",
        "label": "XGBoost",       "color": "darkorange",  "marker": "s", "ls": "--",
    },
    "lgbm_250": {
        "type": "lgbm", "train_energy": 250,
        "results_dir": os.path.join(BASE_DIR, "LGBM_model2_250GeV"),
        "model_file":  "lgbm_model2_250GeV.txt",
        "scaler_file": "lgbm_model2_scaler.npz",
        "label": "LightGBM",      "color": "forestgreen", "marker": "^", "ls": "-.",
    },
}
 
# Groups to compare
GROUPS = {
    "125GeV": ["rf_125", "xgb_125", "lgbm_125"],
    "250GeV": ["rf_250", "xgb_250", "lgbm_250"],
}

In [3]:
def load_scaler(scaler_path):
    d  = np.load(scaler_path)
    sc = StandardScaler()
    sc.mean_  = d["mean"].astype(np.float64)
    sc.scale_ = d["scale"].astype(np.float64)
    sc.var_   = sc.scale_ ** 2
    sc.n_features_in_ = len(sc.mean_)
    return sc
 
 
def load_model(cfg):
    model_path = os.path.join(cfg["results_dir"], cfg["model_file"])
    mtype = cfg["type"]
    if mtype == "rf":
        return joblib.load(model_path)
    elif mtype == "xgb":
        m = xgb.XGBClassifier()
        m.load_model(model_path)
        return m
    elif mtype == "lgbm":
        booster = lgb.Booster(model_file=model_path)
        class LGBMWrapper:
            def __init__(self, b): self.b = b
            def predict_proba(self, X):
                p = self.b.predict(X)
                return np.column_stack([1 - p, p])
            @property
            def feature_importances_(self):
                imp = self.b.feature_importance(importance_type="gain")
                return imp / imp.sum()
        return LGBMWrapper(booster)
 
 
def optimal_threshold(y_true, y_score):
    precision, recall, thresholds = precision_recall_curve(y_true, y_score)
    denom = precision[:-1] + recall[:-1]
    f1s   = np.where(denom > 0, 2 * precision[:-1] * recall[:-1] / denom, 0.0)
    return float(thresholds[np.argmax(f1s)])
 
 
def bkg_rejection_at_wp(y_true, y_score, sig_eff):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    idx = np.argmin(np.abs(tpr - sig_eff))
    return np.inf if fpr[idx] == 0 else 1.0 / fpr[idx]
 
 
def collect_model_results(model_key):
    """Run inference for one model on all test energies, return result dict."""
    cfg = MODEL_CONFIGS[model_key]
    model_path = os.path.join(cfg["results_dir"], cfg["model_file"])
    if not os.path.exists(model_path):
        print(f"  Not found, skipping: {model_key}")
        return None
 
    print(f"  Loading {cfg['label']} ({cfg['train_energy']} GeV)...")
    model  = load_model(cfg)
    scaler = load_scaler(os.path.join(cfg["results_dir"], cfg["scaler_file"]))
 
    results = {}
    for energy in TEST_ENERGIES:
        test_path = os.path.join(FEATURE_DIR, f"bdt_features_{energy}GeV_test.npz")
        dt     = np.load(test_path, allow_pickle=True)
        X_test = scaler.transform(dt["features"].astype(np.float32))
        y_test = dt["labels"].astype(np.int32)
 
        scores = model.predict_proba(X_test)[:, 1]
        auc    = roc_auc_score(y_test, scores)
        thresh = optimal_threshold(y_test, scores)
        preds  = (scores >= thresh).astype(int)
 
        from sklearn.metrics import f1_score, accuracy_score
        f1  = f1_score(y_test, preds, zero_division=0)
        acc = accuracy_score(y_test, preds)
 
        fpr, tpr, _ = roc_curve(y_test, scores)
        brej = {wp: bkg_rejection_at_wp(y_test, scores, wp) for wp in WORKING_POINTS}
 
        results[energy] = dict(
            auc=auc, f1=f1, accuracy=acc,
            fpr=fpr, tpr=tpr, brej=brej,
        )
 
    # feature importance (at in-dist energy, same for all energies)
    try:
        results["feature_importances"] = model.feature_importances_
    except Exception:
        results["feature_importances"] = None
 
    return results

In [4]:
print("=" * 60)
print("Collecting results for all models...")
print("=" * 60)
 
all_results = {}
for key in MODEL_CONFIGS:
    r = collect_model_results(key)
    if r is not None:
        all_results[key] = r
 

energies = np.array(TEST_ENERGIES)
wp_colors = ["royalblue", "darkorange", "forestgreen", "crimson"]
 
for group_name, group_keys in GROUPS.items():
    train_energy = int(group_name.replace("GeV", ""))
    available    = [k for k in group_keys if k in all_results]
    if not available:
        print(f"\nSkipping group {group_name}: no models available")
        continue
 
    print(f"\n{'='*60}")
    print(f"Comparison plots — {group_name} training group")
    print(f"{'='*60}")
 
    # PLOT A: Background rejection curves — in-dist energy 
    fig, ax = plt.subplots(figsize=(8, 6))
    for key in available:
        cfg = MODEL_CONFIGS[key]
        r   = all_results[key][train_energy]
        valid     = r["fpr"] > 0
        brej_curve = np.where(r["fpr"] > 0, 1.0 / r["fpr"], np.nan)
        ax.plot(r["tpr"][valid], brej_curve[valid],
                color=cfg["color"], ls=cfg["ls"], lw=2.0,
                label=f"{cfg['label']}  AUC={r['auc']:.4f}")
 
    ax.set_yscale("log")
    ax.set_xlabel("Signal Efficiency (TPR)", fontsize=12)
    ax.set_ylabel("Background Rejection (1/FPR)", fontsize=12)
    ax.set_title(f"Model Comparison — {group_name} Training\n"
                 f"Background Rejection Curves (in-dist. test @ {train_energy} GeV)", fontsize=12)
    ax.legend(fontsize=10, loc="upper right")
    ax.set_xlim(0, 1); ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARE_DIR, f"compare_{group_name}_brej_curves.png"), dpi=150)
    plt.close()
    print(f"  Saved: compare_{group_name}_brej_curves.png")
 
    # PLOT B: Background rejection curves — all test energies (grid)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
    axes = axes.flatten()
    for i, energy in enumerate(TEST_ENERGIES):
        ax = axes[i]
        for key in available:
            cfg = MODEL_CONFIGS[key]
            r   = all_results[key][energy]
            valid = r["fpr"] > 0
            brej_curve = np.where(r["fpr"] > 0, 1.0 / r["fpr"], np.nan)
            ax.plot(r["tpr"][valid], brej_curve[valid],
                    color=cfg["color"], ls=cfg["ls"], lw=1.8,
                    label=f"{cfg['label']}")
        ax.set_yscale("log")
        id_marker = " ◄" if energy == train_energy else ""
        ax.set_title(f"{energy} GeV{id_marker}", fontsize=11)
        ax.grid(True, which="both", alpha=0.3)
        ax.set_xlim(0, 1)
        if i in [0, 3]: ax.set_ylabel("BkgRej (1/FPR)", fontsize=10)
        if i in [3, 4, 5]: ax.set_xlabel("Signal Efficiency", fontsize=10)
        if i == 0:
            ax.legend(fontsize=8, loc="upper right")
 
    fig.suptitle(f"Background Rejection Curves — {group_name} Training — All Test Energies",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARE_DIR, f"compare_{group_name}_brej_grid.png"), dpi=150)
    plt.close()
    print(f"  Saved: compare_{group_name}_brej_grid.png")
 
    # PLOT C: AUC vs energy 
    fig, ax = plt.subplots(figsize=(7, 5))
    for key in available:
        cfg  = MODEL_CONFIGS[key]
        aucs = np.array([all_results[key][e]["auc"] for e in TEST_ENERGIES])
        ax.plot(energies, aucs, color=cfg["color"], ls=cfg["ls"],
                marker=cfg["marker"], lw=2, ms=7, label=cfg["label"])
    ax.axvline(train_energy, color="gray", ls=":", lw=1.5, label=f"In-dist. ({train_energy} GeV)")
    ax.set_xlabel("Test Energy  √s [GeV]", fontsize=12)
    ax.set_ylabel("AUC", fontsize=12)
    ax.set_title(f"Model Comparison — {group_name} Training\nAUC vs Test Energy", fontsize=12)
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    all_aucs = [all_results[k][e]["auc"] for k in available for e in TEST_ENERGIES]
    ax.set_ylim(min(all_aucs) - 0.002, 1.001)
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARE_DIR, f"compare_{group_name}_auc_vs_energy.png"), dpi=150)
    plt.close()
    print(f"  Saved: compare_{group_name}_auc_vs_energy.png")
 
    # PLOT D: F1 vs energy 
    fig, ax = plt.subplots(figsize=(7, 5))
    for key in available:
        cfg = MODEL_CONFIGS[key]
        f1s = np.array([all_results[key][e]["f1"] for e in TEST_ENERGIES])
        ax.plot(energies, f1s, color=cfg["color"], ls=cfg["ls"],
                marker=cfg["marker"], lw=2, ms=7, label=cfg["label"])
    ax.axvline(train_energy, color="gray", ls=":", lw=1.5, label=f"In-dist. ({train_energy} GeV)")
    ax.set_xlabel("Test Energy  √s [GeV]", fontsize=12)
    ax.set_ylabel("F1 Score", fontsize=12)
    ax.set_title(f"Model Comparison — {group_name} Training\nF1 Score vs Test Energy", fontsize=12)
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
    all_f1s = [all_results[k][e]["f1"] for k in available for e in TEST_ENERGIES]
    ax.set_ylim(max(0.85, min(all_f1s) - 0.01), 1.001)
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARE_DIR, f"compare_{group_name}_f1_vs_energy.png"), dpi=150)
    plt.close()
    print(f"  Saved: compare_{group_name}_f1_vs_energy.png")
 
    # PLOT E: Background rejection at WPs vs energy
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True)
    axes = axes.flatten()
    for j, wp in enumerate(WORKING_POINTS):
        ax = axes[j]
        for key in available:
            cfg   = MODEL_CONFIGS[key]
            brejs = np.array([all_results[key][e]["brej"][wp] for e in TEST_ENERGIES])
            brejs_plot = np.where(np.isinf(brejs),
                                  np.nanmax(brejs[~np.isinf(brejs)]) * 2 if np.any(~np.isinf(brejs)) else 1e6,
                                  brejs)
            ax.plot(energies, brejs_plot, color=cfg["color"], ls=cfg["ls"],
                    marker=cfg["marker"], lw=1.8, ms=6, label=cfg["label"])
        ax.axvline(train_energy, color="gray", ls=":", lw=1.2, alpha=0.7)
        ax.set_yscale("log")
        ax.set_title(f"BkgRej @ {int(wp*100)}% Signal Efficiency", fontsize=11)
        ax.set_ylabel("Background Rejection (1/FPR)", fontsize=10)
        ax.set_xlabel("Test Energy  √s [GeV]", fontsize=10)
        ax.grid(True, which="both", alpha=0.3)
        if j == 0: ax.legend(fontsize=9)
 
    fig.suptitle(f"Background Rejection at Working Points — {group_name} Training",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARE_DIR, f"compare_{group_name}_brej_workingpoints.png"), dpi=150)
    plt.close()
    print(f"  Saved: compare_{group_name}_brej_workingpoints.png")
 
    # PLOT F: Feature importance comparison
    n_models  = len(available)
    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 6), sharey=True)
    if n_models == 1: axes = [axes]
 
    for ax, key in zip(axes, available):
        cfg = MODEL_CONFIGS[key]
        imp = all_results[key].get("feature_importances")
        if imp is None:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes)
            continue
        sorted_idx = np.argsort(imp)
        bars = ax.barh([FEATURE_NAMES[i] for i in sorted_idx], imp[sorted_idx],
                       color=cfg["color"], alpha=0.8)
        ax.set_xlabel("Feature Importance", fontsize=11)
        ax.set_title(cfg["label"], fontsize=11)
        ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)
        ax.set_xlim(0, imp.max() * 1.15)
        ax.grid(True, axis="x", alpha=0.3)
 
    fig.suptitle(f"Feature Importance Comparison — {group_name} Training",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARE_DIR, f"compare_{group_name}_feature_importance.png"), dpi=150)
    plt.close()
    print(f"  Saved: compare_{group_name}_feature_importance.png")

  Loading Random Forest (125 GeV)...


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 700 out of 700 | elapsed:    2.2s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 700 out of 700 | elapsed:    2.2s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 700 out of 700 | elapsed: 

  Loading XGBoost (125 GeV)...
  Loading LightGBM (125 GeV)...
  Loading Random Forest (250 GeV)...


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 700 out of 700 | elapsed:    2.2s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.4s
[Parallel(n_jobs=10)]: Done 700 out of 700 | elapsed:    2.3s finished
[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done  30 tasks      | elapsed:    0.1s
[Parallel(n_jobs=10)]: Done 180 tasks      | elapsed:    0.6s
[Parallel(n_jobs=10)]: Done 430 tasks      | elapsed:    1.5s
[Parallel(n_jobs=10)]: Done 700 out of 700 | elapsed: 

  Loading XGBoost (250 GeV)...
  Loading LightGBM (250 GeV)...

Comparison plots — 125GeV training group


C:\Users\User\AppData\Local\Temp\ipykernel_9720\516009257.py:32: RuntimeWarning: divide by zero encountered in divide
  brej_curve = np.where(r["fpr"] > 0, 1.0 / r["fpr"], np.nan)


  Saved: compare_125GeV_brej_curves.png


C:\Users\User\AppData\Local\Temp\ipykernel_9720\516009257.py:58: RuntimeWarning: divide by zero encountered in divide
  brej_curve = np.where(r["fpr"] > 0, 1.0 / r["fpr"], np.nan)


  Saved: compare_125GeV_brej_grid.png
  Saved: compare_125GeV_auc_vs_energy.png
  Saved: compare_125GeV_f1_vs_energy.png
  Saved: compare_125GeV_brej_workingpoints.png
  Saved: compare_125GeV_feature_importance.png

Comparison plots — 250GeV training group
  Saved: compare_250GeV_brej_curves.png
  Saved: compare_250GeV_brej_grid.png
  Saved: compare_250GeV_auc_vs_energy.png
  Saved: compare_250GeV_f1_vs_energy.png
  Saved: compare_250GeV_brej_workingpoints.png
  Saved: compare_250GeV_feature_importance.png


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY PLOT — all 6 models in one AUC plot
# ─────────────────────────────────────────────────────────────────────────────
 
if len(all_results) > 0:
    print("\nGenerating all-model summary plots...")
 
    # Colours for 125 vs 250: warm=125, cool=250
    summary_styles = {
        "rf_125":   {"color": "#1f77b4", "ls": "-",  "marker": "o", "label": "RF (125 GeV)"},
        "xgb_125":  {"color": "#ff7f0e", "ls": "-",  "marker": "s", "label": "XGBoost (125 GeV)"},
        "lgbm_125": {"color": "#2ca02c", "ls": "-",  "marker": "^", "label": "LightGBM (125 GeV)"},
        "rf_250":   {"color": "#1f77b4", "ls": "--", "marker": "o", "label": "RF (250 GeV)"},
        "xgb_250":  {"color": "#ff7f0e", "ls": "--", "marker": "s", "label": "XGBoost (250 GeV)"},
        "lgbm_250": {"color": "#2ca02c", "ls": "--", "marker": "^", "label": "LightGBM (250 GeV)"},
    }
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
 
    # Panel A: AUC
    ax = axes[0]
    for key, style in summary_styles.items():
        if key not in all_results: continue
        aucs = np.array([all_results[key][e]["auc"] for e in TEST_ENERGIES])
        ax.plot(energies, aucs, color=style["color"], ls=style["ls"],
                marker=style["marker"], lw=1.8, ms=6, label=style["label"])
    ax.axvline(125, color="silver", ls=":", lw=1.2, label="In-dist. 125 GeV")
    ax.axvline(250, color="darkgray", ls=":", lw=1.2, label="In-dist. 250 GeV")
    ax.set_xlabel("Test Energy  √s [GeV]", fontsize=12)
    ax.set_ylabel("AUC", fontsize=12)
    ax.set_title("Overall AUC vs Energy — All Models", fontsize=12)
    ax.legend(fontsize=8, loc="lower right"); ax.grid(True, alpha=0.3)
 
    # Panel B: F1
    ax = axes[1]
    for key, style in summary_styles.items():
        if key not in all_results: continue
        f1s = np.array([all_results[key][e]["f1"] for e in TEST_ENERGIES])
        ax.plot(energies, f1s, color=style["color"], ls=style["ls"],
                marker=style["marker"], lw=1.8, ms=6, label=style["label"])
    ax.axvline(125, color="silver", ls=":", lw=1.2)
    ax.axvline(250, color="darkgray", ls=":", lw=1.2)
    ax.set_xlabel("Test Energy  √s [GeV]", fontsize=12)
    ax.set_ylabel("F1 Score", fontsize=12)
    ax.set_title("F1 Score vs Energy — All Models", fontsize=12)
    ax.legend(fontsize=8, loc="lower right"); ax.grid(True, alpha=0.3)
 
    fig.suptitle("BDT Model Family Comparison — Cross-Energy Generalisation",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(COMPARE_DIR, "compare_all_models_summary.png"), dpi=150)
    plt.close()
    print("  Saved: compare_all_models_summary.png")
 
    # ── Summary table ─────────────────────────────────────────────────────
    lines = [
        "All-Model Summary — AUC at Each Test Energy",
        "=" * 85,
        f"{'Model':<25}" + "".join([f"  {e}GeV" for e in TEST_ENERGIES]),
        "-" * 85,
    ]
    for key, style in summary_styles.items():
        if key not in all_results: continue
        aucs = [all_results[key][e]["auc"] for e in TEST_ENERGIES]
        lines.append(f"{style['label']:<25}" + "".join([f"  {a:.4f}" for a in aucs]))
    lines.append("=" * 85)
 
    table_str = "\n".join(lines)
    print("\n" + table_str)
    with open(os.path.join(COMPARE_DIR, "compare_all_models_auc_table.txt"), "w", encoding="utf-8") as f:
        f.write(table_str + "\n")
 
print("\n" + "=" * 60)
print(f"All comparison plots saved to: {COMPARE_DIR}")
print("=" * 60)


Generating all-model summary plots...
  Saved: compare_all_models_summary.png

All-Model Summary — AUC at Each Test Energy
Model                      100GeV  125GeV  150GeV  200GeV  250GeV  300GeV
-------------------------------------------------------------------------------------
RF (125 GeV)               0.9979  0.9986  0.9989  0.9992  0.9992  0.9993
XGBoost (125 GeV)          0.9935  0.9957  0.9967  0.9974  0.9976  0.9978
LightGBM (125 GeV)         0.9981  0.9987  0.9991  0.9993  0.9993  0.9994
RF (250 GeV)               0.9968  0.9982  0.9988  0.9994  0.9995  0.9996
XGBoost (250 GeV)          0.9974  0.9984  0.9989  0.9994  0.9996  0.9997
LightGBM (250 GeV)         0.9974  0.9984  0.9989  0.9994  0.9996  0.9997

All comparison plots saved to: E:/Python/MSc_Project_Upgrade/results_analysis/bdt\comparison
